# SUB009 — Neural Coordinate Refinement + Template-Neural Hybrid

**Competition**: Stanford RNA 3D Folding Part 2  
**Metric**: TM-score (higher is better, best-of-5)  
**Lineage**: SUB004 (0.211) → SUB005 → SUB006 (val 0.179) → SUB007 → SUB008 → SUB009

**IT008 key additions** over SUB008:
- Inline `RNATransformerModel` (6-layer, 1.15M params) trained on 2671 competition sequences
- AdamW + CosineAnnealingLR + mixed-precision (AMP) + early stopping
- MC-dropout diversity: 3 stochastic forward passes per test target
- Combined candidate pool: 10 IT007 + 4 neural → max-dispersion selects 5
- Template-neural Kabsch blend for partial-coverage targets (identity 0.1–0.3)
- Sliding-window inference for sequences > 2048 nt (handles 4640 nt outlier)
- Wall-clock training budget: 3 hours; falls back to IT007 if neural fails

In [ ]:
import os, sys, time, zlib, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print("SUB009: Neural Refinement + Template-Neural Hybrid (IT008)")
print("=" * 60)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

INPUT_BASE = "/kaggle/input"
if os.path.isdir(INPUT_BASE):
    print(f"\nContents of {INPUT_BASE}:")
    for item in sorted(os.listdir(INPUT_BASE)):
        full = os.path.join(INPUT_BASE, item)
        if os.path.isdir(full):
            sub_items = os.listdir(full)
            print(f"  {item}/ ({len(sub_items)} items)")
            for si in sorted(sub_items)[:10]:
                print(f"    {si}")
        else:
            print(f"  {item} ({os.path.getsize(full)} bytes)")


In [ ]:
CANDIDATE_BASES = [
    "/kaggle/input/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-part-2",
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
]

DATA_PATH = None
for p in CANDIDATE_BASES:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "test_sequences.csv")):
        DATA_PATH = p
        break

if DATA_PATH is None and os.path.isdir(INPUT_BASE):
    for root, dirs, files in os.walk(INPUT_BASE):
        if "test_sequences.csv" in files:
            DATA_PATH = root
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find competition data. "
        f"Searched: {CANDIDATE_BASES}. "
        f"Available: {os.listdir(INPUT_BASE) if os.path.isdir(INPUT_BASE) else 'N/A'}"
    )

print(f"\nUsing data path: {DATA_PATH}")
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

train_seqs   = pd.read_csv(os.path.join(DATA_PATH, "train_sequences.csv"))
test_seqs    = pd.read_csv(os.path.join(DATA_PATH, "test_sequences.csv"))
train_labels = pd.read_csv(os.path.join(DATA_PATH, "train_labels.csv"))
sample_sub   = pd.read_csv(os.path.join(DATA_PATH, "sample_submission.csv"))

val_seq_path = os.path.join(DATA_PATH, "validation_sequences.csv")
val_lab_path = os.path.join(DATA_PATH, "validation_labels.csv")
HAS_VAL = os.path.isfile(val_seq_path) and os.path.isfile(val_lab_path)
if HAS_VAL:
    val_seqs   = pd.read_csv(val_seq_path)
    val_labels = pd.read_csv(val_lab_path)
    print(f"Validation set loaded: {len(val_seqs)} targets")
else:
    val_seqs = val_labels = None
    print("No validation set found")

print(f"Train sequences: {len(train_seqs)}")
print(f"Test sequences:  {len(test_seqs)}")
print(f"Train labels:    {len(train_labels)}")
print(f"Sample sub:      {len(sample_sub)}")


In [ ]:
# ============================================================
# Build Template Library from Training + Validation Data
# ============================================================
SENTINEL = -1e18

def process_labels(labels_df):
    coords_dict = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for id_prefix, group in labels_df.groupby(prefixes, sort=False):
        sorted_group = group.sort_values("resid")
        c = sorted_group[["x_1", "y_1", "z_1"]].values.astype(np.float64)
        mask = np.all(np.abs(c - SENTINEL) < 1e10, axis=1)
        if mask.any():
            c[mask] = np.nan
        coords_dict[id_prefix] = c
    return coords_dict

def build_template_bank(seqs_df, coords_dict, min_valid=5, min_valid_frac=0.0):
    ids, seqs, coords_list, lens, valid_fracs = [], [], [], [], []
    for r in seqs_df.itertuples(index=False):
        tid = r.target_id
        if tid not in coords_dict:
            continue
        seq = r.sequence
        coords = coords_dict[tid]
        if len(coords) != len(seq):
            continue
        valid_mask = ~np.isnan(coords[:, 0])
        n_valid = valid_mask.sum()
        if n_valid < min_valid:
            continue
        if np.all(coords[valid_mask] == 0):
            continue
        vf = float(n_valid) / len(seq)
        if vf < min_valid_frac:
            continue
        ids.append(tid)
        seqs.append(seq)
        coords_list.append(coords)
        lens.append(len(seq))
        valid_fracs.append(vf)
    return ids, seqs, coords_list, np.array(lens, dtype=np.int32), np.array(valid_fracs, dtype=np.float32)

print("Processing training labels...")
t0 = time.time()
train_coords_dict = process_labels(train_labels)
print(f"Processed {len(train_coords_dict)} targets in {time.time()-t0:.1f}s")

TRAIN_IDS, TRAIN_SEQS, TRAIN_COORDS, TRAIN_LENS, TRAIN_VALID_FRAC = \
    build_template_bank(train_seqs, train_coords_dict)
print(f"Train template library: {len(TRAIN_IDS)} templates")

val_coords_dict = {}
if HAS_VAL:
    print("Processing validation labels...")
    val_coords_dict = process_labels(val_labels)
    v_ids, v_seqs, v_coords, v_lens, v_vfracs = \
        build_template_bank(val_seqs, val_coords_dict)
    n_before = len(TRAIN_IDS)
    existing_ids = set(TRAIN_IDS)
    for i in range(len(v_ids)):
        if v_ids[i] not in existing_ids:
            TRAIN_IDS.append(v_ids[i])
            TRAIN_SEQS.append(v_seqs[i])
            TRAIN_COORDS.append(v_coords[i])
    TRAIN_LENS = np.array([len(s) for s in TRAIN_SEQS], dtype=np.int32)
    TRAIN_VALID_FRAC = np.array([
        float(np.sum(~np.isnan(c[:, 0]))) / len(s)
        for s, c in zip(TRAIN_SEQS, TRAIN_COORDS)
    ], dtype=np.float32)
    print(f"Added {len(TRAIN_IDS) - n_before} validation templates")

print(f"Combined template library: {len(TRAIN_IDS)} templates")
print(f"  Length range: {TRAIN_LENS.min()}-{TRAIN_LENS.max()}, mean={TRAIN_LENS.mean():.0f}")
print(f"  Valid fraction: mean={TRAIN_VALID_FRAC.mean():.2f}, median={np.median(TRAIN_VALID_FRAC):.2f}")


In [ ]:
# ============================================================
# K-mer Indexing for Fast Template Retrieval
# ============================================================
KMER_K = 5
PREFILTER_TOP = 400
ALIGN_TOP = 60

_BASE2 = {"A": 0, "C": 1, "G": 2, "U": 3}

def kmer_set_2bit(seq, k=KMER_K):
    if len(seq) < k:
        return frozenset()
    mask = (1 << (2 * k)) - 1
    code = 0
    out = set()
    valid = True
    for i in range(k):
        b = _BASE2.get(seq[i])
        if b is None:
            valid = False
            break
        code = ((code << 2) | b) & mask
    if valid:
        out.add(code)
        for i in range(k, len(seq)):
            b = _BASE2.get(seq[i])
            if b is None:
                continue
            code = ((code << 2) | b) & mask
            out.add(code)
    return frozenset(out)

print("Building k-mer index...")
t0 = time.time()
TRAIN_KMERS = [kmer_set_2bit(s, KMER_K) for s in TRAIN_SEQS]
print(f"K-mer index built for {len(TRAIN_KMERS)} templates in {time.time()-t0:.1f}s")


In [ ]:
# ============================================================
# Nussinov Secondary Structure Prediction
# ============================================================
_SS_SCORES = {
    ('A', 'U'): 2, ('U', 'A'): 2,
    ('G', 'C'): 3, ('C', 'G'): 3,
    ('G', 'U'): 1, ('U', 'G'): 1,
}
_MIN_LOOP = 4

def nussinov_predict(seq, max_len=1500):
    L = len(seq)
    if L > max_len:
        return _nussinov_windowed(seq, window=1000, overlap=200)
    dp = np.zeros((L, L), dtype=np.int16)
    for span in range(_MIN_LOOP + 1, L):
        for i in range(L - span):
            j = i + span
            dp[i, j] = dp[i + 1, j]
            dp[i, j] = max(dp[i, j], dp[i, j - 1])
            sc = _SS_SCORES.get((seq[i], seq[j]), 0)
            if sc > 0:
                dp[i, j] = max(dp[i, j], dp[i + 1, j - 1] + sc)
            for k in range(i + 1, j):
                dp[i, j] = max(dp[i, j], dp[i, k] + dp[k + 1, j])
    pairs = []
    _traceback(dp, seq, 0, L - 1, pairs)
    return pairs

def _traceback(dp, seq, i, j, pairs):
    if i >= j - _MIN_LOOP:
        return
    if dp[i, j] == dp[i + 1, j]:
        _traceback(dp, seq, i + 1, j, pairs)
    elif dp[i, j] == dp[i, j - 1]:
        _traceback(dp, seq, i, j - 1, pairs)
    else:
        sc = _SS_SCORES.get((seq[i], seq[j]), 0)
        if sc > 0 and dp[i, j] == dp[i + 1, j - 1] + sc:
            pairs.append((i, j))
            _traceback(dp, seq, i + 1, j - 1, pairs)
        else:
            for k in range(i + 1, j):
                if dp[i, j] == dp[i, k] + dp[k + 1, j]:
                    _traceback(dp, seq, i, k, pairs)
                    _traceback(dp, seq, k + 1, j, pairs)
                    break

def _nussinov_windowed(seq, window=1000, overlap=200):
    L = len(seq)
    all_pairs = []
    step = window - overlap
    pos = 0
    while pos < L:
        end = min(pos + window, L)
        sub = seq[pos:end]
        local_pairs = nussinov_predict(sub, max_len=window + 100)
        for pi, pj in local_pairs:
            all_pairs.append((pi + pos, pj + pos))
        pos += step
    return list(set(all_pairs))

_ss_cache = {}

def get_secondary_structure(seq):
    if seq not in _ss_cache:
        _ss_cache[seq] = nussinov_predict(seq)
    return _ss_cache[seq]

print("Nussinov secondary structure prediction loaded.")
test_ss = nussinov_predict("GCAUUAGCUC")
print(f"  Test: GCAUUAGCUC -> {len(test_ss)} pairs: {test_ss}")


In [ ]:
# ============================================================
# Alignment Functions with RNA-specific scoring
# ============================================================
_MATCH = 2
_MISMATCH_TRANSITION = -0.5
_MISMATCH_TRANSVERSION = -1.5
_GAP_OPEN = -4
_GAP_EXTEND = -1

_PURINES    = {'A', 'G'}
_PYRIMIDINES = {'C', 'U'}

def _mismatch_score(a, b):
    if a == b:
        return _MATCH
    if (a in _PURINES and b in _PURINES) or (a in _PYRIMIDINES and b in _PYRIMIDINES):
        return _MISMATCH_TRANSITION
    return _MISMATCH_TRANSVERSION

def needleman_wunsch(seq_a, seq_b):
    n, m = len(seq_a), len(seq_b)
    if n * m > 25_000_000:
        return _banded_nw(seq_a, seq_b, band_width=150)
    dp = np.full((n + 1, m + 1), -999999.0, dtype=np.float32)
    dp[0, 0] = 0.0
    for j in range(1, m + 1):
        dp[0, j] = _GAP_OPEN + (j - 1) * _GAP_EXTEND
    for i in range(1, n + 1):
        dp[i, 0] = _GAP_OPEN + (i - 1) * _GAP_EXTEND
    b_arr = np.array(list(seq_b))
    for i in range(1, n + 1):
        ch_a = seq_a[i - 1]
        scores = np.array([_mismatch_score(ch_a, ch_b) for ch_b in b_arr], dtype=np.float32)
        diag = dp[i - 1, :-1] + scores
        up = dp[i - 1, 1:] + _GAP_EXTEND
        dp[i, 1:] = np.maximum(diag, up)
        for j in range(1, m + 1):
            left = dp[i, j - 1] + _GAP_EXTEND
            dp[i, j] = max(dp[i, j], left)
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = _mismatch_score(seq_a[i-1], seq_b[j-1])
            if abs(dp[i, j] - (dp[i-1, j-1] + s)) < 1e-3:
                map_a_to_b[i - 1] = j - 1
                i -= 1; j -= 1
                continue
        if i > 0 and abs(dp[i, j] - (dp[i-1, j] + _GAP_EXTEND)) < 1e-3:
            i -= 1
        elif j > 0:
            j -= 1
        else:
            break
    return map_a_to_b, float(dp[n, m])

def _banded_nw(seq_a, seq_b, band_width=150):
    n, m = len(seq_a), len(seq_b)
    NEGINF = -999999.0
    dp = {}
    dp[(0, 0)] = 0.0
    for j in range(1, min(band_width + 1, m + 1)):
        dp[(0, j)] = _GAP_OPEN + (j - 1) * _GAP_EXTEND
    for i in range(1, min(band_width + 1, n + 1)):
        dp[(i, 0)] = _GAP_OPEN + (i - 1) * _GAP_EXTEND
    ratio = m / n if n > 0 else 1.0
    for i in range(1, n + 1):
        center_j = int(i * ratio)
        j_lo = max(1, center_j - band_width)
        j_hi = min(m, center_j + band_width)
        for j in range(j_lo, j_hi + 1):
            s = _mismatch_score(seq_a[i-1], seq_b[j-1])
            d = dp.get((i-1, j-1), NEGINF) + s
            u = dp.get((i-1, j), NEGINF) + _GAP_EXTEND
            l = dp.get((i, j-1), NEGINF) + _GAP_EXTEND
            dp[(i, j)] = max(d, u, l)
    score = dp.get((n, m), NEGINF)
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = _mismatch_score(seq_a[i-1], seq_b[j-1])
            if abs(dp.get((i, j), NEGINF) - (dp.get((i-1, j-1), NEGINF) + s)) < 1e-3:
                map_a_to_b[i - 1] = j - 1
                i -= 1; j -= 1
                continue
        if i > 0 and abs(dp.get((i, j), NEGINF) - (dp.get((i-1, j), NEGINF) + _GAP_EXTEND)) < 1e-3:
            i -= 1
        elif j > 0:
            j -= 1
        else:
            break
    return map_a_to_b, float(score)

def nw_score_only(seq_a, seq_b):
    n, m = len(seq_a), len(seq_b)
    if n * m > 50_000_000:
        return _kmer_score_proxy(seq_a, seq_b)
    prev = np.full(m + 1, -999999.0, dtype=np.float32)
    prev[0] = 0.0
    for j in range(1, m + 1):
        prev[j] = _GAP_OPEN + (j - 1) * _GAP_EXTEND
    b_arr = list(seq_b)
    for i in range(1, n + 1):
        curr = np.empty(m + 1, dtype=np.float32)
        curr[0] = _GAP_OPEN + (i - 1) * _GAP_EXTEND
        ch_a = seq_a[i - 1]
        scores = np.array([_mismatch_score(ch_a, ch_b) for ch_b in b_arr], dtype=np.float32)
        diag = prev[:-1] + scores
        up = prev[1:] + _GAP_EXTEND
        curr[1:] = np.maximum(diag, up)
        for j in range(1, m + 1):
            curr[j] = max(curr[j], curr[j - 1] + _GAP_EXTEND)
        prev = curr
    return float(prev[m])

def _kmer_score_proxy(seq_a, seq_b, k=5):
    ka = kmer_set_2bit(seq_a, k)
    kb = kmer_set_2bit(seq_b, k)
    if not ka or not kb:
        return 0.0
    inter = len(ka & kb)
    union = len(ka | kb)
    jaccard = inter / union if union > 0 else 0.0
    return jaccard * 2.0 * min(len(seq_a), len(seq_b))

print("Alignment functions loaded.")


In [ ]:
# ============================================================
# Kabsch Superposition and Coordinate Transfer
# ============================================================
def kabsch_superpose(P, Q):
    assert P.shape == Q.shape and P.shape[1] == 3
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    P_c = P - centroid_P
    Q_c = Q - centroid_Q
    H = P_c.T @ Q_c
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign_matrix = np.diag([1.0, 1.0, np.sign(d)])
    R = Vt.T @ sign_matrix @ U.T
    t = centroid_Q - R @ centroid_P
    return R, t

def transfer_coordinates(query_seq, template_seq, template_coords, alignment_map):
    qlen = len(query_seq)
    result = np.full((qlen, 3), np.nan, dtype=np.float64)
    n_transferred = 0
    for q_pos, t_pos in alignment_map.items():
        if q_pos < qlen and t_pos < len(template_coords):
            c = template_coords[t_pos]
            if not np.any(np.isnan(c)):
                result[q_pos] = c
                n_transferred += 1
    coverage = n_transferred / qlen if qlen > 0 else 0
    mapped = sorted([k for k in range(qlen) if not np.isnan(result[k, 0])])
    if not mapped:
        return _helical_chain(qlen), 0.0
    for i in range(qlen):
        if not np.isnan(result[i, 0]):
            continue
        prev_v = next((j for j in range(i - 1, -1, -1) if not np.isnan(result[j, 0])), -1)
        next_v = next((j for j in range(i + 1, qlen) if not np.isnan(result[j, 0])), -1)
        if prev_v >= 0 and next_v >= 0:
            gap_len = next_v - prev_v
            pos_in_gap = i - prev_v
            w = pos_in_gap / gap_len
            base = (1 - w) * result[prev_v] + w * result[next_v]
            if gap_len > 2:
                direction = result[next_v] - result[prev_v]
                norm_d = np.linalg.norm(direction)
                if norm_d > 1e-6:
                    perp = np.cross(direction, [0, 0, 1])
                    perp_norm = np.linalg.norm(perp)
                    if perp_norm < 1e-6:
                        perp = np.cross(direction, [0, 1, 0])
                        perp_norm = np.linalg.norm(perp)
                    perp = perp / (perp_norm + 1e-12)
                    perp2 = np.cross(direction / norm_d, perp)
                    angle = 2 * np.pi * pos_in_gap / max(gap_len, 1)
                    radius = min(2.0, 0.5 * gap_len)
                    helix_offset = radius * (np.cos(angle) * perp + np.sin(angle) * perp2)
                    dampen = np.sin(np.pi * w)
                    base += helix_offset * dampen
            result[i] = base
        elif prev_v >= 0:
            if prev_v > 0 and not np.isnan(result[prev_v - 1, 0]):
                d = result[prev_v] - result[prev_v - 1]
                d = d / (np.linalg.norm(d) + 1e-12) * 5.95
            else:
                d = np.array([5.95, 0, 0])
            result[i] = result[prev_v] + d * (i - prev_v)
        elif next_v >= 0:
            if next_v < qlen - 1 and not np.isnan(result[next_v + 1, 0]):
                d = result[next_v + 1] - result[next_v]
                d = d / (np.linalg.norm(d) + 1e-12) * 5.95
            else:
                d = np.array([5.95, 0, 0])
            result[i] = result[next_v] - d * (next_v - i)
        else:
            result[i] = [i * 5.95, 0, 0]
    return np.nan_to_num(result), coverage

def _helical_chain(length, spacing=5.95, twist_deg=32.7, radius=2.0):
    coords = np.zeros((length, 3), dtype=np.float64)
    twist_rad = np.deg2rad(twist_deg)
    rise = spacing * 0.47
    for i in range(length):
        angle = i * twist_rad
        coords[i, 0] = radius * np.cos(angle)
        coords[i, 1] = radius * np.sin(angle)
        coords[i, 2] = i * rise
    return coords

print("Coordinate transfer loaded.")


In [ ]:
# ============================================================
# Coverage-Weighted Template Retrieval
# ============================================================
def find_templates_for_chain(chain_seq, top_n=ALIGN_TOP, length_tolerance=0.50):
    qlen = len(chain_seq)
    qkm = kmer_set_2bit(chain_seq, KMER_K)
    qkm_len = len(qkm)
    if len(TRAIN_IDS) == 0:
        return []
    maxL = np.maximum(TRAIN_LENS, qlen)
    keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= length_tolerance
    idxs = np.where(keep)[0]
    if idxs.size < 10:
        keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= 0.70
        idxs = np.where(keep)[0]
    if idxs.size == 0:
        diffs = np.abs(TRAIN_LENS - qlen)
        idxs = np.argsort(diffs)[:PREFILTER_TOP]
    scored = []
    for i in idxs:
        tkm = TRAIN_KMERS[i]
        if qkm_len == 0 or len(tkm) == 0:
            jac = 0.0
        else:
            inter = len(qkm & tkm)
            union = qkm_len + len(tkm) - inter
            jac = inter / union if union else 0.0
        len_ratio = min(qlen, TRAIN_LENS[i]) / max(qlen, TRAIN_LENS[i])
        valid_frac = TRAIN_VALID_FRAC[i]
        composite = jac * np.sqrt(len_ratio) * valid_frac
        scored.append((composite, jac, int(i)))
    scored.sort(key=lambda x: x[0], reverse=True)
    top_candidates = scored[:PREFILTER_TOP]
    NW_CELL_LIMIT = 5_000_000
    sims = []
    for composite, jac, i in top_candidates:
        t_seq = TRAIN_SEQS[i]
        if qlen * len(t_seq) < NW_CELL_LIMIT:
            raw = nw_score_only(chain_seq, t_seq)
            denom = 2.0 * min(qlen, len(t_seq))
            norm = float(raw / denom) if denom > 0 else jac
        else:
            norm = jac
        len_ratio = min(qlen, len(t_seq)) / max(qlen, len(t_seq))
        final_score = norm * np.sqrt(len_ratio) * TRAIN_VALID_FRAC[i]
        sims.append((TRAIN_IDS[i], t_seq, final_score, norm, TRAIN_COORDS[i], i))
    sims.sort(key=lambda x: x[2], reverse=True)
    return sims[:top_n]

print("Template retrieval loaded.")


In [ ]:
# ============================================================
# Fragment-Based Assembly
# ============================================================
FRAGMENT_SIZE = 100
FRAGMENT_OVERLAP = 30
FRAGMENT_MIN_SIM = 0.15

def fragment_assembly(chain_seq, fallback_coords=None):
    L = len(chain_seq)
    if L <= FRAGMENT_SIZE:
        cands = find_templates_for_chain(chain_seq, top_n=5)
        if cands:
            t_id, t_seq, score, sim, t_coords, idx = cands[0]
            amap, _ = needleman_wunsch(chain_seq, t_seq)
            coords, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
            return coords, cov
        return _helical_chain(L), 0.0
    step = FRAGMENT_SIZE - FRAGMENT_OVERLAP
    fragments = []
    pos = 0
    while pos < L:
        end = min(pos + FRAGMENT_SIZE, L)
        if L - end < FRAGMENT_OVERLAP and end < L:
            end = L
        fragments.append((pos, end))
        if end >= L:
            break
        pos += step
    frag_coords = []
    frag_coverages = []
    for frag_start, frag_end in fragments:
        frag_seq = chain_seq[frag_start:frag_end]
        cands = find_templates_for_chain(frag_seq, top_n=5, length_tolerance=0.60)
        if cands and cands[0][2] > FRAGMENT_MIN_SIM:
            t_id, t_seq, score, sim, t_coords, idx = cands[0]
            amap, _ = needleman_wunsch(frag_seq, t_seq)
            coords, cov = transfer_coordinates(frag_seq, t_seq, t_coords, amap)
            frag_coords.append(coords)
            frag_coverages.append(cov)
        else:
            frag_coords.append(None)
            frag_coverages.append(0.0)
    result = np.full((L, 3), np.nan, dtype=np.float64)
    placed_mask = np.zeros(L, dtype=bool)
    for fi, (frag_start, frag_end) in enumerate(fragments):
        if frag_coords[fi] is None:
            continue
        fc = frag_coords[fi]
        if fi == 0:
            result[frag_start:frag_end] = fc
            placed_mask[frag_start:frag_end] = True
        else:
            overlap_start = frag_start
            prev_end = fragments[fi-1][1]
            overlap_end = min(prev_end, frag_end)
            overlap_len = overlap_end - overlap_start
            if overlap_len >= 4 and placed_mask[overlap_start:overlap_end].sum() >= 4:
                ref_pts = result[overlap_start:overlap_end]
                new_pts = fc[:overlap_len]
                valid = ~(np.isnan(ref_pts[:, 0]) | np.isnan(new_pts[:, 0]))
                if valid.sum() >= 4:
                    try:
                        R, t = kabsch_superpose(new_pts[valid], ref_pts[valid])
                        fc_aligned = (fc @ R.T) + t
                        result[overlap_end:frag_end] = fc_aligned[overlap_len:]
                        placed_mask[overlap_end:frag_end] = True
                        for k in range(overlap_start, overlap_end):
                            if placed_mask[k] and not np.isnan(fc_aligned[k - frag_start, 0]):
                                w = (k - overlap_start) / max(overlap_len - 1, 1)
                                result[k] = (1 - w) * result[k] + w * fc_aligned[k - frag_start]
                        continue
                    except Exception:
                        pass
            for k in range(frag_start, frag_end):
                if not placed_mask[k]:
                    result[k] = fc[k - frag_start]
                    placed_mask[k] = True
    if fallback_coords is not None:
        for i in range(L):
            if np.isnan(result[i, 0]):
                result[i] = fallback_coords[i]
    else:
        for i in range(L):
            if np.isnan(result[i, 0]):
                result[i] = [i * 5.95, 0, 0]
    coverage = placed_mask.sum() / L
    return np.nan_to_num(result), coverage

print("Fragment assembly loaded.")


In [ ]:
# ============================================================
# Stoichiometry and Chain Handling
# ============================================================
def parse_fasta(fasta_content):
    out = {}
    cur = None
    seq_parts = []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(seq_parts)
            cur = line[1:].split()[0]
            seq_parts = []
        else:
            seq_parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(seq_parts)
    return out

def parse_stoichiometry(stoich):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    out = []
    for part in str(stoich).split(";"):
        parts = part.split(":")
        if len(parts) == 2:
            out.append((parts[0].strip(), int(parts[1])))
    return out

def get_chain_info(row):
    seq = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_seq = row.get("all_sequences", "")
    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip() == "" or str(all_seq).strip() == "":
        return [(0, len(seq), seq)]
    try:
        chain_dict = parse_fasta(all_seq)
        order = parse_stoichiometry(stoich)
        chains = []
        pos = 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq), seq)]
            for _ in range(cnt):
                L = len(base)
                chains.append((pos, pos + L, base))
                pos += L
        if pos != len(seq):
            return [(0, len(seq), seq)]
        return chains
    except Exception:
        return [(0, len(seq), seq)]

test_chain_info = {}
for _, r in test_seqs.iterrows():
    test_chain_info[r["target_id"]] = get_chain_info(r)

print(f"Chain info computed for {len(test_chain_info)} test targets")
for tid, chains in list(test_chain_info.items())[:5]:
    total_res = sum(e-s for s,e,_ in chains)
    unique_seqs = len(set(cs for _,_,cs in chains))
    print(f"  {tid}: {len(chains)} chain(s), {unique_seqs} unique, total {total_res} residues")


In [ ]:
# ============================================================
# Distance Geometry De Novo Folding (IT007)
# ============================================================
def distance_geometry_fold(seq, pairs, seed=42):
    rng = np.random.default_rng(seed)
    L = len(seq)
    if L < 3:
        return _helical_chain(L)
    pair_set = set()
    pair_map = {}
    for i, j in pairs:
        if i < L and j < L:
            pair_set.add((min(i,j), max(i,j)))
            pair_map.setdefault(i, []).append(j)
            pair_map.setdefault(j, []).append(i)
    BOND_DIST = 5.95
    SKIP1_DIST = 10.2
    BP_DIST = 10.5
    rows, cols, dists = [], [], []
    for i in range(L - 1):
        rows.append(i); cols.append(i+1); dists.append(BOND_DIST)
    for i in range(L - 2):
        rows.append(i); cols.append(i+2); dists.append(SKIP1_DIST)
    for i, j in pair_set:
        rows.append(i); cols.append(j); dists.append(BP_DIST)
    for i, j in pair_set:
        if (min(i+1, j-1), max(i+1, j-1)) in pair_set:
            rows.append(i); cols.append(i+1); dists.append(3.4)
    if not rows:
        return _helical_chain(L)
    rows = np.array(rows, dtype=np.int32)
    cols = np.array(cols, dtype=np.int32)
    dists = np.array(dists, dtype=np.float64)
    D2 = np.full((L, L), 0.0, dtype=np.float64)
    for r, c, d in zip(rows, cols, dists):
        D2[r, c] = d * d
        D2[c, r] = d * d
    for i in range(L):
        for j in range(i+1, min(i+8, L)):
            if D2[i, j] == 0:
                est_dist = BOND_DIST * (j - i) * 0.85
                D2[i, j] = est_dist * est_dist
                D2[j, i] = D2[i, j]
    for i in range(L):
        for j in range(i+8, L):
            if D2[i, j] == 0:
                sep = j - i
                est_dist = min(BOND_DIST * sep * 0.5, BOND_DIST * L * 0.3)
                D2[i, j] = est_dist * est_dist
                D2[j, i] = D2[i, j]
    n = L
    if n > 500:
        return _mds_subsampled(L, D2, pairs, rng)
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ D2 @ H
    try:
        eigenvalues, eigenvectors = np.linalg.eigh(B)
        idx = np.argsort(eigenvalues)[::-1][:3]
        vals = np.maximum(eigenvalues[idx], 0.0)
        vecs = eigenvectors[:, idx]
        xyz = vecs * np.sqrt(vals)[None, :]
    except Exception:
        return _helical_chain(L)
    xyz = _enforce_backbone(xyz, BOND_DIST, iterations=10)
    return xyz

def _mds_subsampled(L, D2, pairs, rng, n_anchor=200):
    indices = np.linspace(0, L-1, min(n_anchor, L)).astype(int)
    indices = np.unique(indices)
    n = len(indices)
    D2_sub = D2[np.ix_(indices, indices)]
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ D2_sub @ H
    try:
        eigenvalues, eigenvectors = np.linalg.eigh(B)
        idx = np.argsort(eigenvalues)[::-1][:3]
        vals = np.maximum(eigenvalues[idx], 0.0)
        vecs = eigenvectors[:, idx]
        anchor_xyz = vecs * np.sqrt(vals)[None, :]
    except Exception:
        return _helical_chain(L)
    xyz = np.zeros((L, 3), dtype=np.float64)
    for i in range(L):
        pos = np.searchsorted(indices, i)
        if pos >= len(indices):
            xyz[i] = anchor_xyz[-1]
        elif pos == 0 or indices[pos] == i:
            xyz[i] = anchor_xyz[min(pos, n-1)]
        else:
            i0, i1 = indices[pos-1], indices[pos]
            w = (i - i0) / max(i1 - i0, 1)
            xyz[i] = (1 - w) * anchor_xyz[pos-1] + w * anchor_xyz[pos]
    xyz = _enforce_backbone(xyz, 5.95, iterations=10)
    return xyz

def _enforce_backbone(xyz, target_dist, iterations=10):
    L = len(xyz)
    for _ in range(iterations):
        for i in range(L - 1):
            d = xyz[i+1] - xyz[i]
            dist = np.linalg.norm(d)
            if dist < 1e-6:
                d = np.array([target_dist, 0, 0], dtype=np.float64)
                dist = target_dist
            err = (target_dist - dist) / dist
            adj = d * err * 0.3
            xyz[i] -= adj
            xyz[i+1] += adj
    return xyz

print("Distance geometry de novo folding loaded.")


In [ ]:
# ============================================================
# Simulated Annealing Coordinate Refinement (IT007)
# ============================================================
def sa_refine_coordinates(coordinates, segments, chain_seqs, confidence=1.0, iterations=30):
    coords = coordinates.copy()
    T_init = 1.0 * (1.0 - min(confidence, 0.95))
    T_init = max(T_init, 0.05)
    T_final = 0.005
    for it in range(iterations):
        T = T_init * (T_final / T_init) ** (it / max(iterations - 1, 1))
        step_size = 0.3 * T / T_init
        for seg_idx, (seg_s, seg_e) in enumerate(segments):
            X = coords[seg_s:seg_e]
            L = seg_e - seg_s
            if L < 3:
                continue
            d = X[1:] - X[:-1]
            dist = np.linalg.norm(d, axis=1, keepdims=True) + 1e-8
            target = 5.95
            force = (d / dist) * (target - dist) * step_size * 0.5
            X[:-1] -= force
            X[1:] += force
            if L >= 5:
                d2 = X[2:] - X[:-2]
                dist2 = np.linalg.norm(d2, axis=1, keepdims=True) + 1e-8
                force2 = (d2 / dist2) * (10.2 - dist2) * step_size * 0.2
                X[:-2] -= force2
                X[2:] += force2
            if L >= 6:
                d3 = X[3:] - X[:-3]
                dist3 = np.linalg.norm(d3, axis=1, keepdims=True) + 1e-8
                force3 = (d3 / dist3) * (13.5 - dist3) * step_size * 0.08
                X[:-3] -= force3
                X[3:] += force3
            if seg_idx < len(chain_seqs) and L <= 2000:
                chain_seq = chain_seqs[seg_idx]
                pairs = get_secondary_structure(chain_seq)
                for pi, pj in pairs:
                    if pi < L and pj < L:
                        diff = X[pj] - X[pi]
                        dist_bp = np.linalg.norm(diff) + 1e-8
                        if dist_bp > 4.0:
                            err = (10.5 - dist_bp) / dist_bp
                            bp_force = diff * err * step_size * 0.15
                            X[pi] -= bp_force
                            X[pj] += bp_force
                pair_map = {}
                for pi, pj in pairs:
                    pair_map[pi] = pj
                    pair_map[pj] = pi
                for pi, pj in pairs:
                    if pi + 1 in pair_map and pair_map[pi + 1] == pj - 1:
                        if pi + 1 < L and 0 <= pj - 1 < L:
                            d_stack = X[pi + 1] - X[pi]
                            dist_stack = np.linalg.norm(d_stack) + 1e-8
                            if dist_stack > 2.0:
                                err_s = (3.4 - dist_stack) / dist_stack
                                sf = d_stack * err_s * step_size * 0.05
                                X[pi] -= sf
                                X[pi + 1] += sf
            if L >= 5:
                lap = 0.5 * (X[:-2] + X[2:]) - X[1:-1]
                X[1:-1] += step_size * 0.15 * lap
            if L >= 20 and it % 3 == 0:
                k = min(L, 150)
                idx_arr = np.linspace(0, L - 1, k).astype(int) if k < L else np.arange(L)
                P = X[idx_arr]
                diff = P[:, None, :] - P[None, :, :]
                distm = np.linalg.norm(diff, axis=2) + 1e-8
                sep = np.abs(idx_arr[:, None] - idx_arr[None, :])
                mask_clash = (sep > 2) & (distm < 3.2)
                if np.any(mask_clash):
                    force_clash = (3.2 - distm) / distm
                    vec = (diff * force_clash[:, :, None] * mask_clash[:, :, None]).sum(axis=1)
                    X[idx_arr] += step_size * 0.05 * vec
            coords[seg_s:seg_e] = X
    return coords

print("Simulated annealing refinement loaded.")


In [ ]:
# ============================================================
# Template Blending with Kabsch
# ============================================================
def blend_templates(query_seq, templates, weights):
    qlen = len(query_seq)
    if not templates:
        return _helical_chain(qlen)
    transfers = []
    for t_seq, t_coords, amap, sim in templates:
        coords, cov = transfer_coordinates(query_seq, t_seq, t_coords, amap)
        transfers.append(coords)
    if len(transfers) == 1:
        return transfers[0]
    ref = transfers[0]
    aligned = [ref]
    for k in range(1, len(transfers)):
        other = transfers[k]
        valid = ~(np.isnan(ref[:, 0]) | np.isnan(other[:, 0]))
        n_valid = valid.sum()
        if n_valid >= 4:
            try:
                R, t = kabsch_superpose(other[valid], ref[valid])
                other_aligned = (other @ R.T) + t
                aligned.append(other_aligned)
            except Exception:
                aligned.append(other)
        else:
            aligned.append(other)
    w = np.array(weights[:len(aligned)], dtype=np.float64)
    w = w / (w.sum() + 1e-12)
    result = np.zeros((qlen, 3), dtype=np.float64)
    for k in range(len(aligned)):
        result += w[k] * aligned[k]
    return result

print("Template blending loaded.")


In [ ]:
# ============================================================
# SS-Guided De Novo with Distance Geometry Fallback (IT007)
# ============================================================
def ss_denovo_fold(seq, seed=42):
    L = len(seq)
    if L < 3:
        return _helical_chain(L)
    pairs = get_secondary_structure(seq) if L <= 2000 else []
    if pairs and L <= 500:
        xyz = distance_geometry_fold(seq, pairs, seed=seed)
    else:
        rng = np.random.default_rng(seed)
        xyz = np.zeros((L, 3), dtype=np.float64)
        pair_map = {}
        for i, j in pairs:
            pair_map[i] = j
            pair_map[j] = i
        direction = np.array([1.0, 0.0, 0.0], dtype=np.float64)
        bond_len = 5.95
        twist = np.deg2rad(32.7)
        for i in range(1, L):
            angle = twist + rng.normal(0, 0.08)
            axis = np.array([0, 0, 1], dtype=np.float64) + rng.normal(0, 0.1, 3)
            axis = axis / (np.linalg.norm(axis) + 1e-12)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            ux, uy, uz = axis
            C = 1 - cos_a
            R = np.array([
                [cos_a + ux*ux*C, ux*uy*C - uz*sin_a, ux*uz*C + uy*sin_a],
                [uy*ux*C + uz*sin_a, cos_a + uy*uy*C, uy*uz*C - ux*sin_a],
                [uz*ux*C - uy*sin_a, uz*uy*C + ux*sin_a, cos_a + uz*uz*C]
            ])
            direction = R @ direction
            direction = direction / (np.linalg.norm(direction) + 1e-12)
            if i in pair_map:
                partner = pair_map[i]
                if partner < i:
                    toward = xyz[partner] - xyz[i-1]
                    target_dist = 10.5
                    cur_dist = np.linalg.norm(toward)
                    if cur_dist > 1e-6:
                        bias = toward / cur_dist
                        w_bp = min(0.5, target_dist / (cur_dist + 1e-6))
                        direction = (1 - w_bp) * direction + w_bp * bias
                        direction = direction / (np.linalg.norm(direction) + 1e-12)
            xyz[i] = xyz[i-1] + direction * bond_len
        for _ in range(8):
            for pi, pj in pairs:
                if pi < L and pj < L:
                    diff = xyz[pj] - xyz[pi]
                    dist = np.linalg.norm(diff) + 1e-8
                    err = (10.5 - dist) / dist
                    adj = diff * err * 0.2
                    xyz[pi] -= adj
                    xyz[pj] += adj
            for i in range(L - 1):
                d = xyz[i+1] - xyz[i]
                dist = np.linalg.norm(d) + 1e-8
                err = (bond_len - dist) / dist
                adj = d * err * 0.25
                xyz[i] -= adj
                xyz[i+1] += adj
    return xyz

print("SS-guided de novo folding loaded (IT007: distance geometry).")


In [ ]:
# ============================================================
# TM-score Computation
# ============================================================
def compute_tm_score(pred_coords, true_coords):
    valid = ~(np.isnan(true_coords[:, 0]) | np.isnan(pred_coords[:, 0]))
    if valid.sum() < 3:
        return 0.0
    pred = pred_coords[valid]
    true = true_coords[valid]
    L = len(true)
    if L < 15:
        d0 = 0.5
    else:
        d0 = 0.6 * np.sqrt(L - 0.5) - 2.5
        d0 = max(d0, 0.5)
    try:
        R, t = kabsch_superpose(pred, true)
        pred_aligned = (pred @ R.T) + t
    except Exception:
        pred_aligned = pred
    distances = np.linalg.norm(pred_aligned - true, axis=1)
    tm = np.sum(1.0 / (1.0 + (distances / d0) ** 2)) / L
    return float(tm)

def compute_best_of_5_tm(all_preds, true_coords):
    best = 0.0
    for pred in all_preds:
        tm = compute_tm_score(pred, true_coords)
        best = max(best, tm)
    return best

print("TM-score computation loaded.")


In [ ]:
# ============================================================
# Max-Dispersion Diversity Selection (IT007)
# ============================================================
def compute_rmsd(coords1, coords2):
    diff = coords1 - coords2
    return np.sqrt(np.mean(np.sum(diff**2, axis=1)))

def select_diverse_predictions(candidates, n_select=5):
    if len(candidates) <= n_select:
        return list(candidates)
    n = len(candidates)
    rmsd_matrix = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i+1, n):
            r = compute_rmsd(candidates[i], candidates[j])
            rmsd_matrix[i, j] = r
            rmsd_matrix[j, i] = r
    selected = [0]
    remaining = set(range(1, n))
    for _ in range(n_select - 1):
        if not remaining:
            break
        best_idx = -1
        best_min_dist = -1
        for idx in remaining:
            min_dist = min(rmsd_matrix[idx, s] for s in selected)
            if min_dist > best_min_dist:
                best_min_dist = min_dist
                best_idx = idx
        if best_idx >= 0:
            selected.append(best_idx)
            remaining.discard(best_idx)
    return [candidates[i] for i in selected]

print("Max-dispersion diversity selection loaded.")


In [ ]:
# ============================================================
# Per-Chain Multi-Strategy Prediction Pipeline (IT007)
# ============================================================
N_CANDIDATES = 10

def predict_single_chain_raw(chain_seq, chain_id_str):
    """Return all N_CANDIDATES predictions before diversity selection."""
    L = len(chain_seq)
    cands = find_templates_for_chain(chain_seq, top_n=ALIGN_TOP)
    candidates = []

    if not cands:
        for pi in range(N_CANDIDATES):
            seed = (zlib.adler32(f"{chain_id_str}_{pi}".encode()) & 0xFFFFFFFF)
            candidates.append(ss_denovo_fold(chain_seq, seed=seed))
        return candidates, 0.0

    aligned_templates = []
    for c_id, c_seq, c_score, c_sim, c_coords, c_idx in cands[:min(15, len(cands))]:
        amap, score = needleman_wunsch(chain_seq, c_seq)
        aligned_templates.append((c_id, c_seq, c_coords, amap, c_sim, score))

    best_sim = aligned_templates[0][4] if aligned_templates else 0.0
    use_fragments = (L > 200 and best_sim < 0.40)

    for i in range(N_CANDIDATES):
        try:
            if i == 0:
                _, t_seq, t_coords, amap, sim, _ = aligned_templates[0]
                X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
            elif i == 1 and use_fragments:
                _, t_seq, t_coords, amap, sim, _ = aligned_templates[0]
                fallback, _ = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
                X, cov = fragment_assembly(chain_seq, fallback_coords=fallback)
            elif i < len(aligned_templates) + 1 and i <= 5:
                t_idx = min(i, len(aligned_templates) - 1)
                _, t_seq, t_coords, amap, sim, _ = aligned_templates[t_idx]
                X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
            elif i == 6 and len(aligned_templates) >= 3:
                blend_input = []
                blend_weights = []
                for j in range(min(3, len(aligned_templates))):
                    _, t_seq, t_coords, amap, sim, _ = aligned_templates[j]
                    blend_input.append((t_seq, t_coords, amap, sim))
                    blend_weights.append(max(sim, 0.01))
                X = blend_templates(chain_seq, blend_input, blend_weights)
            elif i == 7 and len(aligned_templates) >= 5:
                blend_input = []
                blend_weights = []
                for j in range(min(5, len(aligned_templates))):
                    _, t_seq, t_coords, amap, sim, _ = aligned_templates[j]
                    blend_input.append((t_seq, t_coords, amap, sim))
                    blend_weights.append(max(sim, 0.01))
                X = blend_templates(chain_seq, blend_input, blend_weights)
            elif i >= 8:
                seed = (zlib.adler32(f"{chain_id_str}_dg_{i}".encode()) & 0xFFFFFFFF)
                X = ss_denovo_fold(chain_seq, seed=seed)
            else:
                t_idx = min(i % len(aligned_templates), len(aligned_templates) - 1)
                _, t_seq, t_coords, amap, sim, _ = aligned_templates[t_idx]
                X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
                seed = (zlib.adler32(f"{chain_id_str}_{i}".encode()) & 0xFFFFFFFF)
                rng = np.random.default_rng(seed)
                noise_std = max(0.3, (0.5 - sim) * 2.0)
                X = X + rng.normal(0, noise_std, X.shape)
            candidates.append(X)
        except Exception:
            seed = (zlib.adler32(f"{chain_id_str}_err_{i}".encode()) & 0xFFFFFFFF)
            candidates.append(ss_denovo_fold(chain_seq, seed=seed))

    return candidates, best_sim

def predict_single_chain(chain_seq, chain_id_str, n_predictions=5):
    candidates, _ = predict_single_chain_raw(chain_seq, chain_id_str)
    return select_diverse_predictions(candidates, n_predictions)

def predict_complex(tid, full_seq, chain_info, n_predictions=5):
    L = len(full_seq)
    all_predictions = [np.zeros((L, 3), dtype=np.float64) for _ in range(n_predictions)]
    chain_cache = {}
    chain_seqs = []
    for ci, (start, end, chain_seq) in enumerate(chain_info):
        chain_seqs.append(chain_seq)
        if chain_seq in chain_cache:
            chain_preds = chain_cache[chain_seq]
        else:
            chain_id = f"{tid}_chain{ci}"
            chain_preds = predict_single_chain(chain_seq, chain_id, n_predictions=n_predictions)
            chain_cache[chain_seq] = chain_preds
        for pi in range(n_predictions):
            pred_coords = chain_preds[pi] if pi < len(chain_preds) else chain_preds[-1]
            chain_len = end - start
            if ci > 0:
                seed = (zlib.adler32(f"{tid}_{ci}_{pi}".encode()) & 0xFFFFFFFF)
                rng = np.random.default_rng(seed)
                axis = rng.normal(size=3)
                axis = axis / (np.linalg.norm(axis) + 1e-12)
                angle = rng.uniform(0, 2 * np.pi)
                c, s = np.cos(angle), np.sin(angle)
                x, y, z = axis
                C = 1.0 - c
                R = np.array([
                    [c + x*x*C, x*y*C - z*s, x*z*C + y*s],
                    [y*x*C + z*s, c + y*y*C, y*z*C - x*s],
                    [z*x*C - y*s, z*y*C + x*s, c + z*z*C]
                ])
                centroid = pred_coords.mean(axis=0)
                rotated = (pred_coords - centroid) @ R.T + centroid
                offset = rng.normal(0, 15, size=3)
                pred_coords = rotated + offset
            all_predictions[pi][start:end] = pred_coords[:chain_len]
    segments = [(s, e) for s, e, _ in chain_info]
    for pi in range(n_predictions):
        all_predictions[pi] = sa_refine_coordinates(
            all_predictions[pi], segments, chain_seqs, confidence=0.5, iterations=25)
    return all_predictions

print("Per-chain prediction pipeline loaded (IT007: diversity selection + SA refinement).")


In [ ]:
# ============================================================
# IT008 — Neural Coordinate Refinement
# ============================================================
#
# Strategy:
#   1. Train RNATransformerModel (6-layer, 1.15M params) on 2671 training sequences
#   2. At inference: combine IT007 (10 candidates) + neural (det + 3 MC-dropout) = 14 total
#   3. Max-dispersion selects 5 from the combined pool
#   4. Neural model fills the gap for 22/28 test targets with no template coverage
#
# Key design choices:
#   - use_pair_bias=False: disables O(L^2) PairRepresentation → linear memory
#   - max_len=4096: covers all training sequences
#   - BucketBatchSampler: avoids OOM from variable-length batches
#   - MSE loss for first 10 epochs (stable cold-start), then RMSD
#   - 3-hour wall-clock budget; fallback to IT007 if neural fails

print("=" * 60)
print("IT008: Neural Coordinate Refinement")
print("=" * 60)


In [ ]:
# ============================================================
# Inline RNATransformerModel (use_pair_bias=False)
# ============================================================
import math

# Nucleotide tokenization: pad=0, A=1, C=2, G=3, U=4, T=4
NUC_TO_IDX_NEURAL = {"A": 1, "C": 2, "G": 3, "U": 4, "T": 4}
NUM_TOKENS = 5  # 0=pad, 1-4=nucleotides

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=4096, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class PreNormLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.nhead = nhead
        self.d_model = d_model
        self.head_dim = d_model // nhead
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout),
        )
        self.attn_dropout = nn.Dropout(dropout)
        self.res_dropout = nn.Dropout(dropout)

    def _attn(self, x, key_padding_mask=None):
        B, L, D = x.shape
        q = self.q_proj(x).reshape(B, L, self.nhead, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).reshape(B, L, self.nhead, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).reshape(B, L, self.nhead, self.head_dim).transpose(1, 2)
        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        if key_padding_mask is not None:
            attn_mask = ~key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn = attn.masked_fill(attn_mask, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, L, D)
        return self.out_proj(out)

    def forward(self, x, key_padding_mask=None):
        x = x + self.res_dropout(self._attn(self.norm1(x), key_padding_mask))
        x = x + self.ffn(self.norm2(x))
        return x


class RNATransformerNeural(nn.Module):
    """Lightweight RNA Transformer for IT008 in-kernel training.
    
    use_pair_bias is intentionally disabled to keep memory linear in L.
    max_len=4096 covers all training sequences.
    """
    def __init__(self, num_tokens=5, d_model=128, nhead=8, num_layers=6,
                 dim_feedforward=512, dropout=0.1, max_len=4096):
        super().__init__()
        self.d_model = d_model
        self.token_embed = nn.Embedding(num_tokens, d_model, padding_idx=0)
        self.pos_enc = SinusoidalPE(d_model, max_len, dropout)
        self.input_norm = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList([
            PreNormLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.coord_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 3),
        )

    def forward(self, seq_indices, mask=None):
        """seq_indices: (B, L) long; mask: (B, L) bool (True=valid); returns (B, L, 3)"""
        x = self.token_embed(seq_indices)
        x = self.pos_enc(x)
        x = self.input_norm(x)
        for layer in self.layers:
            x = layer(x, key_padding_mask=mask)
        x = self.final_norm(x)
        return self.coord_head(x)

# Quick sanity check
_test_model = RNATransformerNeural()
_n_params = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
print(f"RNATransformerNeural: {_n_params:,} parameters")
_test_seq = torch.randint(1, 5, (2, 50))
_test_mask = torch.ones(2, 50, dtype=torch.bool)
_test_out = _test_model(_test_seq, _test_mask)
assert _test_out.shape == (2, 50, 3), f"Expected (2,50,3) got {_test_out.shape}"
print("Forward pass OK:", _test_out.shape)
del _test_model, _test_seq, _test_mask, _test_out


In [ ]:
# ============================================================
# Neural Training Data Preparation
# ============================================================
NEURAL_MAX_TRAIN_LEN = 2048  # truncate longer sequences during training

def seq_to_indices(seq, max_len=None):
    """Convert nucleotide string to tensor of indices."""
    if max_len is not None:
        seq = seq[:max_len]
    return torch.tensor([NUC_TO_IDX_NEURAL.get(c.upper(), 0) for c in seq], dtype=torch.long)

def build_neural_samples(seqs, coords_list, max_len=NEURAL_MAX_TRAIN_LEN, min_valid=5):
    """Build (seq_str, coords_np, valid_mask_np) from TRAIN_SEQS/TRAIN_COORDS.
    
    Truncates sequences to max_len. Sentinel mask: valid = ~isnan(coords).
    Returns list of dicts: {seq, coords, valid_mask, length}
    """
    samples = []
    for seq, coords in zip(seqs, coords_list):
        # Truncate
        if len(seq) > max_len:
            seq = seq[:max_len]
            coords = coords[:max_len]
        L = len(seq)
        valid_mask = ~np.isnan(coords[:, 0])  # (L,) bool
        if valid_mask.sum() < min_valid:
            continue
        # Replace NaN with 0 (masked out during loss computation)
        coords_clean = np.where(valid_mask[:, None], coords, 0.0).astype(np.float32)
        samples.append({
            "seq": seq,
            "coords": coords_clean,
            "valid_mask": valid_mask,
            "length": L,
        })
    return samples

print("Building neural training samples from TRAIN_SEQS/TRAIN_COORDS...")
t0 = time.time()
# Use only first N_TRAIN_TEMPLATES (= original train, not val) for training
# (val templates were appended after index n_train_original)
# We'll use all of TRAIN_SEQS since val seqs won't be in test set
neural_train_samples = build_neural_samples(TRAIN_SEQS, TRAIN_COORDS)
print(f"Neural training samples: {len(neural_train_samples)} in {time.time()-t0:.1f}s")

# Build validation samples if available
neural_val_samples = []
if HAS_VAL:
    v_seqs_list  = list(val_seqs["sequence"])
    v_ids_list   = list(val_seqs["target_id"])
    v_coords_list = [val_coords_dict.get(tid) for tid in v_ids_list]
    valid_pairs = [(s, c) for s, c in zip(v_seqs_list, v_coords_list) if c is not None and len(c) == len(s)]
    v_seqs_f  = [p[0] for p in valid_pairs]
    v_coords_f = [p[1] for p in valid_pairs]
    neural_val_samples = build_neural_samples(v_seqs_f, v_coords_f)
    print(f"Neural validation samples: {len(neural_val_samples)}")

# Length distribution
lengths = [s["length"] for s in neural_train_samples]
print(f"Length stats: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.0f}, median={np.median(lengths):.0f}")
for thresh in [100, 250, 500, 1000, 2048]:
    n = sum(1 for l in lengths if l <= thresh)
    print(f"  L <= {thresh}: {n} ({100*n/len(lengths):.1f}%)")


In [ ]:
# ============================================================
# BucketBatchSampler + Loss Functions
# ============================================================
import random

class BucketBatchSampler:
    """Groups samples by length bucket to minimize padding waste."""
    
    BUCKET_BATCH_SIZES = [
        (100,   16),
        (250,    8),
        (500,    4),
        (1000,   2),
        (2048,   1),
        (99999,  1),
    ]
    
    def __init__(self, samples, shuffle=True, seed=42):
        self.samples = samples
        self.shuffle = shuffle
        self.rng = random.Random(seed)
        self._build_buckets()
    
    def _build_buckets(self):
        # Assign each sample to a bucket by length
        buckets = {thresh: [] for thresh, _ in self.BUCKET_BATCH_SIZES}
        for i, s in enumerate(self.samples):
            L = s["length"]
            for thresh, _ in self.BUCKET_BATCH_SIZES:
                if L <= thresh:
                    buckets[thresh].append(i)
                    break
        self.buckets = buckets
    
    def __iter__(self):
        all_batches = []
        for thresh, bs in self.BUCKET_BATCH_SIZES:
            indices = list(self.buckets[thresh])
            if self.shuffle:
                self.rng.shuffle(indices)
            for start in range(0, len(indices), bs):
                batch = indices[start:start + bs]
                if batch:
                    all_batches.append(batch)
        if self.shuffle:
            self.rng.shuffle(all_batches)
        yield from all_batches
    
    def __len__(self):
        total = 0
        for thresh, bs in self.BUCKET_BATCH_SIZES:
            n = len(self.buckets[thresh])
            total += (n + bs - 1) // bs
        return total

def collate_neural(batch_indices, samples):
    """Collate a list of sample indices into a padded batch."""
    batch = [samples[i] for i in batch_indices]
    max_len = max(s["length"] for s in batch)
    B = len(batch)
    
    seq_t   = torch.zeros(B, max_len, dtype=torch.long)
    coords_t = torch.zeros(B, max_len, 3, dtype=torch.float32)
    pad_mask = torch.zeros(B, max_len, dtype=torch.bool)
    valid_t  = torch.zeros(B, max_len, dtype=torch.bool)
    
    for i, s in enumerate(batch):
        L = s["length"]
        seq_t[i, :L]    = seq_to_indices(s["seq"])
        coords_t[i, :L] = torch.from_numpy(s["coords"])
        pad_mask[i, :L] = True
        valid_t[i, :L]  = torch.from_numpy(s["valid_mask"])
    
    return seq_t, coords_t, pad_mask, valid_t

def rmsd_loss_masked(pred, target, mask):
    """Mean RMSD over batch, with combined padding+sentinel mask.
    pred, target: (B, L, 3); mask: (B, L) bool
    """
    sq_diff = (pred - target) ** 2
    per_res = sq_diff.sum(dim=-1)  # (B, L)
    per_res = per_res * mask.float()
    n_valid = mask.float().sum(dim=-1).clamp(min=1.0)
    msd = per_res.sum(dim=-1) / n_valid
    rmsd = torch.sqrt(msd + 1e-8)
    return rmsd.mean()

def mse_loss_masked(pred, target, mask):
    """MSE loss with combined mask (more stable early in training)."""
    sq_diff = (pred - target) ** 2
    sq_diff = sq_diff * mask.unsqueeze(-1).float()
    n_valid = mask.float().sum().clamp(min=1.0) * 3
    return sq_diff.sum() / n_valid

print("BucketBatchSampler and loss functions loaded.")


In [ ]:
# ============================================================
# Neural Training Loop (IT008)
# ============================================================
NEURAL_CFG = {
    "d_model":         128,
    "nhead":           8,
    "num_layers":      6,
    "dim_feedforward": 512,
    "dropout":         0.1,
    "max_len":         4096,
    "learning_rate":   3e-4,
    "weight_decay":    1e-5,
    "gradient_clip":   1.0,
    "num_epochs":      60,
    "patience":        12,
    "warmup_epochs":   5,
    "use_amp":         True,
    "mse_warmup_epochs": 10,  # use MSE loss for first N epochs, then switch to RMSD
    "wall_clock_budget_s": 3 * 3600,  # 3-hour training budget
}

def run_epoch(model, samples, sampler_or_None, optimizer, scaler, cfg, device, train=True):
    """Run one epoch. Returns mean loss."""
    use_amp = cfg["use_amp"] and device.type == "cuda"
    total_loss = 0.0
    n_batches = 0
    oom_count = 0
    epoch_num = getattr(run_epoch, "_epoch", 0)

    if train:
        sampler = sampler_or_None
    else:
        # Validation: process each sample individually
        model.eval()
        with torch.no_grad():
            for sample in samples:
                seq_t, coords_t, pad_mask, valid_t = collate_neural([0], [sample])
                seq_t    = seq_t.to(device)
                coords_t = coords_t.to(device)
                combined = (pad_mask & valid_t).to(device)
                if combined.sum() == 0:
                    continue
                with torch.cuda.amp.autocast(enabled=use_amp):
                    preds = model(seq_t, mask=pad_mask.to(device))
                    loss  = rmsd_loss_masked(preds, coords_t, combined)
                if torch.isfinite(loss):
                    total_loss += loss.item()
                    n_batches  += 1
        return total_loss / max(n_batches, 1)

    # Training
    model.train()
    for batch_indices in sampler:
        try:
            seq_t, coords_t, pad_mask, valid_t = collate_neural(batch_indices, samples)
            seq_t    = seq_t.to(device)
            coords_t = coords_t.to(device)
            combined = (pad_mask & valid_t).to(device)
            if combined.sum() == 0:
                continue
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                preds = model(seq_t, mask=pad_mask.to(device))
                # Use MSE for first mse_warmup_epochs, then RMSD
                if epoch_num <= cfg["mse_warmup_epochs"]:
                    loss = mse_loss_masked(preds, coords_t, combined)
                else:
                    loss = rmsd_loss_masked(preds, coords_t, combined)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            n_batches  += 1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            oom_count += 1
            if oom_count > 50:
                print(f"    WARNING: {oom_count} OOM errors, consider reducing batch sizes")
            continue
        except Exception as e:
            continue

    return total_loss / max(n_batches, 1)

def train_neural_model(model, train_samples, val_samples, cfg, device):
    """Train the neural model with early stopping and LR scheduling.
    Returns (trained_model, best_val_loss, neural_ok).
    """
    use_amp = cfg["use_amp"] and device.type == "cuda"
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"]
    )
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(cfg["num_epochs"] - cfg["warmup_epochs"], 1),
        eta_min=1e-6,
    )
    
    best_val_loss   = float("inf")
    best_state      = None
    patience_counter = 0
    train_start     = time.time()
    
    for epoch in range(1, cfg["num_epochs"] + 1):
        run_epoch._epoch = epoch
        
        # LR warm-up
        if epoch <= cfg["warmup_epochs"]:
            lr_scale = epoch / cfg["warmup_epochs"]
            for pg in optimizer.param_groups:
                pg["lr"] = cfg["learning_rate"] * lr_scale
        
        sampler = BucketBatchSampler(train_samples, shuffle=True, seed=epoch)
        train_loss = run_epoch(model, train_samples, sampler, optimizer, scaler, cfg, device, train=True)
        
        if val_samples:
            val_loss = run_epoch(model, val_samples, None, optimizer, scaler, cfg, device, train=False)
        else:
            val_loss = train_loss  # no val set: use train loss for early stopping
        
        if epoch > cfg["warmup_epochs"]:
            scheduler.step()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        if epoch % 5 == 0 or epoch == 1:
            elapsed = time.time() - train_start
            print(f"  Epoch {epoch:3d}: train={train_loss:.4f} val={val_loss:.4f} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e} "
                  f"patience={patience_counter} elapsed={elapsed:.0f}s")
        
        if patience_counter >= cfg["patience"]:
            print(f"  Early stopping at epoch {epoch} (patience={cfg['patience']})")
            break
        
        if time.time() - train_start > cfg["wall_clock_budget_s"]:
            print(f"  Wall-clock budget exceeded at epoch {epoch}, stopping.")
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    neural_ok = (best_val_loss < 100.0) and torch.isfinite(torch.tensor(best_val_loss))
    return model, best_val_loss, neural_ok

print("Training loop loaded.")


In [ ]:
# ============================================================
# Train Neural Model
# ============================================================
print("Initializing RNATransformerNeural...")
neural_model = RNATransformerNeural(
    num_tokens=NUM_TOKENS,
    d_model=NEURAL_CFG["d_model"],
    nhead=NEURAL_CFG["nhead"],
    num_layers=NEURAL_CFG["num_layers"],
    dim_feedforward=NEURAL_CFG["dim_feedforward"],
    dropout=NEURAL_CFG["dropout"],
    max_len=NEURAL_CFG["max_len"],
).to(DEVICE)
n_params = sum(p.numel() for p in neural_model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

print(f"\nTraining on {len(neural_train_samples)} samples, validating on {len(neural_val_samples)} samples")
print(f"Device: {DEVICE}, AMP: {NEURAL_CFG['use_amp']}")
print(f"Wall-clock budget: {NEURAL_CFG['wall_clock_budget_s']//3600}h")
print()

t_train_start = time.time()
try:
    neural_model, best_val_loss, NEURAL_OK = train_neural_model(
        neural_model, neural_train_samples, neural_val_samples, NEURAL_CFG, DEVICE
    )
    t_train_total = time.time() - t_train_start
    print(f"\nTraining complete in {t_train_total:.0f}s ({t_train_total/60:.1f}min)")
    print(f"Best validation loss: {best_val_loss:.4f}")
    print(f"Neural model OK: {NEURAL_OK}")
except Exception as e:
    print(f"Neural training failed: {e}")
    NEURAL_OK = False
    best_val_loss = float("inf")

if not NEURAL_OK:
    print("WARNING: Neural model failed. Falling back to pure IT007 pipeline.")
else:
    print("Neural model trained successfully. Will use in prediction pipeline.")
    neural_model.eval()


In [ ]:
# ============================================================
# Neural Inference Functions (IT008)
# ============================================================
NEURAL_INFER_MAX_LEN = 2048  # sliding window for longer sequences

def neural_predict_seq(seq, model, device, mc_dropout=False, max_len=NEURAL_INFER_MAX_LEN):
    """Run neural model on a single sequence.
    
    mc_dropout=True: keep model in train() mode for stochastic dropout (diversity).
    For sequences > max_len, uses sliding window with cosine blending.
    """
    L = len(seq)
    
    if L > max_len:
        return _neural_sliding_window(seq, model, device, mc_dropout, max_len)
    
    if mc_dropout:
        model.train()
    else:
        model.eval()
    
    seq_t = seq_to_indices(seq).unsqueeze(0).to(device)  # (1, L)
    mask  = torch.ones(1, L, dtype=torch.bool, device=device)
    
    ctx = torch.no_grad() if not mc_dropout else torch.enable_grad()
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            coords = model(seq_t, mask=mask)  # (1, L, 3)
    
    model.eval()  # restore eval mode
    return coords[0].cpu().detach().numpy().astype(np.float64)

def _neural_sliding_window(seq, model, device, mc_dropout=False, window=2048, overlap=256):
    """Sliding window inference for sequences longer than window."""
    L = len(seq)
    result = np.zeros((L, 3), dtype=np.float64)
    weight = np.zeros(L, dtype=np.float64)
    
    step = window - overlap
    pos = 0
    while pos < L:
        end = min(pos + window, L)
        sub_seq = seq[pos:end]
        sub_len = len(sub_seq)
        
        sub_coords = neural_predict_seq(sub_seq, model, device, mc_dropout=mc_dropout, max_len=window + 1)
        
        # Cosine blending weights in overlap regions
        w = np.ones(sub_len, dtype=np.float64)
        if pos > 0 and overlap > 0:
            ramp_len = min(overlap, sub_len)
            w[:ramp_len] = 0.5 * (1 - np.cos(np.pi * np.arange(ramp_len) / ramp_len))
        if end < L and overlap > 0:
            ramp_len = min(overlap, sub_len)
            w[-ramp_len:] = 0.5 * (1 - np.cos(np.pi * np.arange(ramp_len, 0, -1) / ramp_len))
        
        # Align to already-placed coordinates if possible
        if pos > 0 and weight[pos:pos+overlap].sum() > 0:
            overlap_end = min(pos + overlap, end)
            ref_pts = result[pos:overlap_end] / np.maximum(weight[pos:overlap_end, None], 1e-8)
            new_pts = sub_coords[:overlap_end - pos]
            valid = (weight[pos:overlap_end] > 0.5)
            if valid.sum() >= 4:
                try:
                    R, t = kabsch_superpose(new_pts[valid], ref_pts[valid])
                    sub_coords = (sub_coords @ R.T) + t
                except Exception:
                    pass
        
        result[pos:end] += sub_coords * w[:, None]
        weight[pos:end] += w
        
        if end >= L:
            break
        pos += step
    
    result = result / np.maximum(weight[:, None], 1e-8)
    return result

def get_neural_candidates(seq, model, device, n_mc=3):
    """Get deterministic + MC-dropout neural candidates."""
    candidates = []
    try:
        # Deterministic pass
        det_coords = neural_predict_seq(seq, model, device, mc_dropout=False)
        candidates.append(det_coords)
        # MC-dropout passes
        for _ in range(n_mc):
            mc_coords = neural_predict_seq(seq, model, device, mc_dropout=True)
            candidates.append(mc_coords)
    except Exception as e:
        print(f"    Neural inference error: {e}")
    return candidates

print("Neural inference functions loaded.")


In [ ]:
# ============================================================
# Combined IT007 + IT008 Prediction Pipeline
# ============================================================
def predict_complex_it008(tid, full_seq, chain_info, n_predictions=5):
    """IT008 pipeline: IT007 candidates (10) + neural candidates (4) → diverse 5.
    
    Falls back to predict_complex() if NEURAL_OK is False.
    """
    if not NEURAL_OK:
        return predict_complex(tid, full_seq, chain_info, n_predictions)
    
    L = len(full_seq)
    segments  = [(s, e) for s, e, _ in chain_info]
    chain_seqs = [cs for _, _, cs in chain_info]
    
    # ── Branch A: IT007 candidates (10 per chain, before diversity selection) ──
    it007_raw = [np.zeros((L, 3), dtype=np.float64) for _ in range(N_CANDIDATES)]
    chain_cache = {}
    best_sim_global = 0.0
    
    for ci, (start, end, chain_seq) in enumerate(chain_info):
        if chain_seq in chain_cache:
            raw_preds, best_sim = chain_cache[chain_seq]
        else:
            chain_id = f"{tid}_chain{ci}"
            raw_preds, best_sim = predict_single_chain_raw(chain_seq, chain_id)
            chain_cache[chain_seq] = (raw_preds, best_sim)
        best_sim_global = max(best_sim_global, best_sim)
        
        for pi in range(N_CANDIDATES):
            pred_coords = raw_preds[pi] if pi < len(raw_preds) else raw_preds[-1]
            chain_len   = end - start
            if ci > 0:
                seed = (zlib.adler32(f"{tid}_{ci}_{pi}".encode()) & 0xFFFFFFFF)
                rng = np.random.default_rng(seed)
                axis = rng.normal(size=3)
                axis = axis / (np.linalg.norm(axis) + 1e-12)
                angle = rng.uniform(0, 2 * np.pi)
                c_ang, s_ang = np.cos(angle), np.sin(angle)
                x, y, z = axis
                C = 1.0 - c_ang
                R = np.array([
                    [c_ang + x*x*C,   x*y*C - z*s_ang, x*z*C + y*s_ang],
                    [y*x*C + z*s_ang, c_ang + y*y*C,   y*z*C - x*s_ang],
                    [z*x*C - y*s_ang, z*y*C + x*s_ang, c_ang + z*z*C],
                ])
                centroid = pred_coords.mean(axis=0)
                rotated  = (pred_coords - centroid) @ R.T + centroid
                offset   = rng.normal(0, 15, size=3)
                pred_coords = rotated + offset
            it007_raw[pi][start:end] = pred_coords[:chain_len]
    
    # SA refine all 10 IT007 candidates
    for pi in range(N_CANDIDATES):
        it007_raw[pi] = sa_refine_coordinates(
            it007_raw[pi], segments, chain_seqs, confidence=0.5, iterations=25)
    
    # ── Branch B: Neural candidates ──
    neural_cands = get_neural_candidates(full_seq, neural_model, DEVICE, n_mc=3)
    
    # Light SA refinement on neural predictions
    for i, nc in enumerate(neural_cands):
        # Confidence = 0.2 (low confidence = more movement allowed)
        neural_cands[i] = sa_refine_coordinates(nc, segments, chain_seqs, confidence=0.2, iterations=10)
    
    # ── Template-neural blend for partial coverage targets ──
    # If best_sim is in [0.1, 0.3], create a hybrid blend candidate
    if neural_cands and 0.05 < best_sim_global < 0.3:
        alpha = min(best_sim_global / 0.3, 1.0)
        template_ref = it007_raw[0]  # best IT007 candidate
        neural_ref   = neural_cands[0]
        valid_both   = ~(np.isnan(template_ref[:, 0]) | np.isnan(neural_ref[:, 0]))
        if valid_both.sum() >= 4:
            try:
                R, t = kabsch_superpose(neural_ref[valid_both], template_ref[valid_both])
                neural_aligned = (neural_ref @ R.T) + t
                hybrid = alpha * template_ref + (1 - alpha) * neural_aligned
                neural_cands.append(hybrid)
            except Exception:
                pass
    
    # ── Combine and select 5 most diverse ──
    all_candidates = it007_raw + neural_cands
    return select_diverse_predictions(all_candidates, n_predictions)

print(f"Combined IT007 + IT008 prediction pipeline loaded.")
print(f"NEURAL_OK = {NEURAL_OK}")


In [ ]:
# ============================================================
# Validation (IT008 vs IT007 comparison)
# ============================================================
if HAS_VAL:
    print("\n" + "=" * 60)
    print("Validation: IT007 vs IT008 comparison")
    print("=" * 60)
    
    if not val_coords_dict:
        val_coords_dict = process_labels(val_labels)
    
    val_chain_info = {}
    for _, r in val_seqs.iterrows():
        val_chain_info[r["target_id"]] = get_chain_info(r)
    
    results_it007 = []
    results_it008 = []
    t0 = time.time()
    
    for _, r in val_seqs.iterrows():
        tid = r["target_id"]
        seq = r["sequence"]
        chains = val_chain_info.get(tid, [(0, len(seq), seq)])
        if tid not in val_coords_dict:
            continue
        true_coords = val_coords_dict[tid]
        if len(true_coords) != len(seq):
            continue
        
        # IT007 (baseline)
        preds_it007 = predict_complex(tid, seq, chains, n_predictions=5)
        tm_it007    = compute_best_of_5_tm(preds_it007, true_coords)
        results_it007.append((tid, tm_it007, len(seq)))
        
        # IT008 (neural + template)
        preds_it008 = predict_complex_it008(tid, seq, chains, n_predictions=5)
        tm_it008    = compute_best_of_5_tm(preds_it008, true_coords)
        results_it008.append((tid, tm_it008, len(seq)))
    
    val_time = time.time() - t0
    
    if results_it007:
        s7 = [s[1] for s in results_it007]
        s8 = [s[1] for s in results_it008]
        print(f"\nValidation results ({len(results_it007)} targets, {val_time:.0f}s):")
        print(f"{'Metric':<20} {'IT007':>10} {'IT008':>10} {'Delta':>10}")
        print("-" * 52)
        print(f"{'Mean TM-score':<20} {np.mean(s7):>10.4f} {np.mean(s8):>10.4f} {np.mean(s8)-np.mean(s7):>+10.4f}")
        print(f"{'Median TM-score':<20} {np.median(s7):>10.4f} {np.median(s8):>10.4f} {np.median(s8)-np.median(s7):>+10.4f}")
        print(f"{'Max TM-score':<20} {np.max(s7):>10.4f} {np.max(s8):>10.4f} {np.max(s8)-np.max(s7):>+10.4f}")
        n7_03 = sum(1 for s in s7 if s > 0.3)
        n8_03 = sum(1 for s in s8 if s > 0.3)
        print(f"{'> 0.3 TM':<20} {n7_03:>10} {n8_03:>10} {n8_03-n7_03:>+10}")
        
        print(f"\n  Top 10 by IT008 score:")
        combined = sorted(zip(results_it008, results_it007), key=lambda x: -x[0][1])
        for (tid, tm8, L), (_, tm7, _) in combined[:10]:
            print(f"    {tid}: IT007={tm7:.4f} IT008={tm8:.4f} delta={tm8-tm7:+.4f} L={L}")
        
        print(f"\n  IT008 improved vs IT007: {sum(1 for a,b in zip(s8,s7) if a>b)}/{len(s7)} targets")
        print(f"  IT008 regressed vs IT007: {sum(1 for a,b in zip(s8,s7) if a<b)}/{len(s7)} targets")
    else:
        print("No valid validation targets found")
else:
    print("Skipping validation (no validation set available)")


In [ ]:
# ============================================================
# Generate Predictions for All Test Targets (IT008 Pipeline)
# ============================================================
print(f"\nGenerating predictions for {len(test_seqs)} test targets...")
print(f"Using IT008 combined pipeline (NEURAL_OK={NEURAL_OK})")
start_time = time.time()

dfs = []

for idx, r in enumerate(test_seqs.itertuples(index=False)):
    tid = r.target_id
    seq = r.sequence
    L   = len(seq)
    chains = test_chain_info.get(tid, [(0, L, seq)])
    
    t0 = time.time()
    preds = predict_complex_it008(tid, seq, chains, n_predictions=5)
    elapsed = time.time() - t0
    
    n_chains = len(chains)
    print(f"  [{idx+1}/{len(test_seqs)}] {tid} (L={L}, chains={n_chains}) -> {elapsed:.1f}s")
    
    data = {
        "ID":      [f"{tid}_{j}" for j in range(1, L + 1)],
        "resname": list(seq),
        "resid":   np.arange(1, L + 1, dtype=np.int32),
    }
    for i in range(5):
        data[f"x_{i+1}"] = preds[i][:, 0].astype(np.float32)
        data[f"y_{i+1}"] = preds[i][:, 1].astype(np.float32)
        data[f"z_{i+1}"] = preds[i][:, 2].astype(np.float32)
    
    dfs.append(pd.DataFrame(data))

total_time = time.time() - start_time
print(f"\nAll predictions generated in {total_time:.1f}s")


In [ ]:
# ============================================================
# Build and Save Submission CSV
# ============================================================
sub = pd.concat(dfs, ignore_index=True)

cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, 6) for c in ["x", "y", "z"]]
coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]

# Sanity checks
n_nan  = sub[coord_cols].isna().sum().sum()
n_inf  = np.isinf(sub[coord_cols].values).sum()
n_zero = (sub[coord_cols] == 0).sum().sum()
print(f"Pre-clip: NaN={n_nan}, Inf={n_inf}, Zero={n_zero}")

sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
sub = sub.fillna(0.0)

output_path = os.path.join(WORK, "submission.csv")
sub[cols].to_csv(output_path, index=False)

print(f"Submission saved to {output_path}")
print(f"Shape:    {sub.shape}")
print(f"Expected: {len(sample_sub)} rows")
print(f"NaN values (post-fill): {sub[coord_cols].isna().sum().sum()}")
print()
print(sub.head(3))
print()
print("=" * 60)
print("SUB009 Pipeline Complete (IT008 Neural + IT007 Template)")
print("=" * 60)
print(f"  Neural model OK   : {NEURAL_OK}")
print(f"  Templates used    : {len(TRAIN_IDS)}")
print(f"  Test targets      : {len(test_seqs)}")
print(f"  Submission rows   : {len(sub)}")
print(f"  Training time     : {t_train_total:.0f}s")
print(f"  Inference time    : {total_time:.0f}s")
print("=" * 60)
